In [79]:
#библиотеки
import pandas as pd
import os
import csv
import re
from Bio import SeqIO

In [91]:
#функция для вытаскивания айдишников из файлов

def get_ids(filename):
    ids = []
    with open(filename, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                id_part = line.split(":")[0]
                ids.append(id_part)
    return ids


In [92]:
notannotCDS_ids = get_ids("Astroviridae_15102025_noannotCDS.txt")
notannotCDS_ids
notannotCDS_target_ids = get_ids("Astroviridae_15102025_noannot_targetCDS.txt")
notannotCDS_target_ids
diff = list(set(notannotCDS_target_ids) - set(notannotCDS_ids))
print(diff)

['MW347540', 'OQ709188', 'OQ709193', 'PQ421843', 'PX395425', 'PQ421852', 'PQ421853', 'PQ421845', 'PQ421842', 'MW645022', 'NC_040647', 'PQ421841', 'OM514376', 'PQ421846', 'PQ421848', 'MN841288', 'MW346737', 'PQ150499', 'MH188020', 'PP211223', 'KX290465', 'PX289196', 'OQ709191', 'OQ709190', 'MT568535', 'MZ182271', 'PQ421844', 'PQ421854', 'MZ443626', 'MW924357', 'MW924356', 'MZ182272', 'MW853972', 'MW645021', 'OQ802761', 'PQ421847', 'OQ709194', 'OQ709189', 'PX289197', 'OP413956', 'MW924358', 'PQ055527', 'PX289198', 'OQ709192', 'MZ291967', 'OP413950', 'PQ161557', 'ON932807']


In [47]:
#превращение tsv файла в csv 
def tsv_to_csv(tsv_path, csv_path):
    df = pd.read_csv(tsv_path, sep="\t")
    df.to_csv(csv_path, index=False)

In [ ]:
#добавление в файл id_prefix (айдишник генома)
def process_files(csv_files):
    result_rows = []
    for csv_file in csv_files:
        result_rows = []
        df = pd.read_csv(csv_file)
        df["ID белка"] = df["ID белка"].astype(str)
        mask1 = df["ID белка"].str.contains("_") & ~df["ID белка"].str.contains("\.")
        df.loc[mask1, "id_prefix"] = df.loc[mask1, "ID белка"].str.split("_").str[0]
        mask2 = df["ID белка"].str.contains(".") & ~df["ID белка"].str.contains("_")
        df.loc[mask2, "id_prefix"] = df.loc[mask2, "ID белка"].str.split(".").str[0]
        result_rows.append(df)
        result = pd.concat(result_rows, ignore_index=True)
        result.to_csv(csv_file, index=False)   
process_files(csv_files)

In [65]:
#скрипт для поиска ORFS
def find_ORFs(csv_files, csv_file_orf, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    orf_df = pd.read_csv(csv_file_orf, header=None).fillna('')
    orf_dict = orf_df.set_index(0)[1].to_dict()
    
    for csv_file in csv_files:
        df = pd.read_csv(csv_file)
        all_results = []

        for _, row in df.iterrows():
            description = str(row.get('Описание предсказания', ''))
            matched = False
            if description in orf_dict.keys():
                all_results.append({
                    'ID белка': row.get('ID белка', ''),
                    'Название семейства/белка': row.get('Название семейства/белка', ''),
                    'Описание предсказания': row.get('Описание предсказания', ''),
                    'Источник предсказания': row.get('Источник предсказания', ''),
                    'ORF': orf_dict[description],
                    'id_prefix': row.get('id_prefix', '')
                })
                matched = True

            if not matched:
                all_results.append({
                    'ID белка': row.get('ID белка', ''), 
                    'Название семейства/белка': row.get('Название семейства/белка', ''),
                    'Описание предсказания': row.get('Описание предсказания', ''),
                    'Источник предсказания': row.get('Источник предсказания', ''),
                    'Вхождение из orf': '-',
                    'ORF': '-',
                    'id_prefix': row.get('id_prefix', '')
                })

        results_df = pd.DataFrame(all_results)
        base_name = os.path.basename(csv_file).rsplit(".", 1)[0]
        output_file = os.path.join(output_dir, f"{base_name}_orf_results.csv")
        results_df.to_csv(output_file, index=False)
        print(f"Файл {csv_file} обработан -> {output_file}")


In [66]:
csv_files = ['interpro_выдача/actual_for_191125/output_interpro_new/not_annot_targetCDS_new.csv','interpro_выдача/actual_for_191125/output_interpro_new/not_annot_CDS_nucl_ORFs_merged_output.csv']
csv_file_orf = '../Domain_descriptions.tsv'
output_dir = 'interpro_выдача/actual_for_191125/predicted_orfs_after_interpro_new/second_try'
find_ORFs(csv_files,csv_file_orf, output_dir)

Файл interpro_выдача/actual_for_191125/output_interpro_new/not_annot_targetCDS_new.csv обработан -> interpro_выдача/actual_for_191125/predicted_orfs_after_interpro_new/second_try/not_annot_targetCDS_new_orf_results.csv
Файл interpro_выдача/actual_for_191125/output_interpro_new/not_annot_CDS_nucl_ORFs_merged_output.csv обработан -> interpro_выдача/actual_for_191125/predicted_orfs_after_interpro_new/second_try/not_annot_CDS_nucl_ORFs_merged_output_orf_results.csv


In [105]:
#Добавляем координаты в файл not_annot_CDS (координаты берем из фаста-файла, предсказанного get_orfs())
def add_orf_coords_from_get_orfs(csv_file, fasta_file, output_csv):
    id_to_coords = {}
    id_to_strand = {}

    with open(fasta_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.startswith('>'):
                m = re.match(r'^>(\S+).*?\[(\d+)\s*-\s*(\d+)\]', line)
                if m:
                    protein_id, start, end = m.groups()
                    coords = f"{start}-{end}"
                    id_to_coords.setdefault(protein_id, []).append(coords)
                    strand = "-" if "REVERSE SENSE" in line.upper() else "+"
                    id_to_strand[protein_id] = strand

    with open(csv_file, newline='', encoding='utf-8') as f_in, \
         open(output_csv, 'w', newline='', encoding='utf-8') as f_out:

        reader = csv.DictReader(f_in)
        fieldnames = reader.fieldnames + ['coords', 'strand']
        writer = csv.DictWriter(f_out, fieldnames=fieldnames)
        writer.writeheader()

        for row in reader:
            protein_id = row.get('ID белка', '')
            coords = ','.join(id_to_coords.get(protein_id, []))
            strand = id_to_strand.get(protein_id, '')
            row['coords'] = coords
            row['strand'] = strand
            writer.writerow(row)


In [106]:
add_orf_coords_from_get_orfs(
    csv_file='interpro_выдача/actual_for_191125/predicted_orfs_after_interpro_new/second_try/not_annot_CDS_nucl_ORFs_merged_output_orf_results.csv',
    fasta_file='interpro_input/not_annot_CDS_nucl_ORFs_after_getorfs.fasta',
    output_csv='interpro_выдача/actual_for_191125/predicted_orfs_after_interpro_new/second_try/not_annot_CDS_nucl_ORFs_merged_output_orf_result_coords.csv'
)

In [80]:
#Добавляем координаты в файл not_annot_target_CDS (координаты берем из genbank-файла)
def add_cds_coords_from_genbank(csv_file, genbank_file, output_file):
    cds_by_accession = {}

    for record in SeqIO.parse(genbank_file, "genbank"):
        if "accessions" in record.annotations:
            acc = record.annotations["accessions"][0]
        else:
            acc = record.id

        cds_list = []
        for feature in record.features:
            if feature.type == "CDS":
                start = int(feature.location.start) + 1   # GenBank: 0-based start
                end = int(feature.location.end)
                strand = "-" if "(-)" in str(feature.location) else "+"
                cds_list.append((f"{start}-{end}", strand))
        cds_by_accession[acc] = cds_list

    df = pd.read_csv(csv_file)
    def get_coords(raw):
        raw = str(raw).rstrip("|")
        if "." in raw:
            acc, idx = raw.split(".")
        else:
            acc, idx = raw, "1"
        try:
            idx = int(idx)
        except:
            idx = 1
        if acc in cds_by_accession and idx <= len(cds_by_accession[acc]):
            return pd.Series(cds_by_accession[acc][idx - 1])

        return pd.Series(["", ""])

    df[["coords", "strand"]] = df.iloc[:, 0].apply(get_coords)
    df.to_csv(output_file, index=False)



In [82]:
add_cds_coords_from_genbank(
    csv_file='interpro_выдача/actual_for_191125/predicted_orfs_after_interpro_new/second_try/not_annot_targetCDS_new_orf_results.csv',
    genbank_file="Astroviridae_15102025.gb",
    output_file="interpro_выдача/actual_for_191125/predicted_orfs_after_interpro_new/second_try/not_annot_targetCDS_new_orf_results_coords.csv"
)

In [117]:
#удаляем строки, где в поле ORFs прочерк
def remove_dash_orf(input_csv, output_csv):
    df = pd.read_csv(input_csv)
    df_clean = df[df['ORF'] != '-']
    df_clean.to_csv(output_csv, index=False)

In [121]:
remove_dash_orf(
    "interpro_выдача/actual_for_191125/predicted_orfs_after_interpro_new/second_try/not_annot_targetCDS_new_orf_results_coords.csv",
    "interpro_выдача/actual_for_191125/predicted_orfs_after_interpro_new/second_try/not_annot_targetCDS_final.csv"
)
remove_dash_orf(
    "interpro_выдача/actual_for_191125/predicted_orfs_after_interpro_new/second_try/not_annot_CDS_nucl_ORFs_merged_output_orf_result_coords.csv",
    "interpro_выдача/actual_for_191125/predicted_orfs_after_interpro_new/second_try/not_annot_CDS_final.csv"
)

In [134]:
#функция на анализ успешных находок и конфликтов 
def analyze_findings(csv_file,target_ids,output_file):
    valid_orf={'1A', '1B', '2'}
    df = pd.read_csv(csv_file)
    res = {}
    other = []

    for _, row in df.iterrows():
        orf_val = str(row["ORF"]).strip()
        if orf_val == '-':
            continue
        if orf_val not in valid_orf and orf_val != '-':
            other.append({
                "ac": row["id_prefix"],
                "orf": orf_val,
                "coords": row["coords"],
                "strand": row["strand"]
            })
            continue
        row_dict = {orf_val: [row["coords"], row["strand"]]}
        res.setdefault(row["id_prefix"], [])
        if row_dict not in res[row["id_prefix"]]:
            res[row["id_prefix"]].append(row_dict)
            
    acc_conflicts = []
    for acc, entries in res.items():
        seen_orf = {}
        seen_coords = {}
        for d in entries:
            (orf, coords) = list(d.items())[0]
            if orf in seen_orf:
                acc_conflicts.append(acc)
            else:
                seen_orf[orf] = True
            if coords[0] in seen_coords:
                acc_conflicts.append(acc)
            else:
                seen_coords[coords[0]] = True

    acc_conflicts = sorted(set(acc_conflicts))

    rows = []
    acc_successful = []
    for acc, ent in res.items():
        if acc not in acc_conflicts and acc in target_ids:
            acc_successful.append(acc)
            for orf_dict in ent:
                for orf, coord in orf_dict.items():
                    rows.append({
                        "ac": acc,
                        "orf": orf,
                        "Coord": coord[0],
                        "Strand": coord[1]
                    })

    successful_df = pd.DataFrame(rows)

    target_ids = set(target_ids)
    found_ids = set(res.keys())
    not_found = sorted(list(target_ids - found_ids))

    conflict_filtered = [x for x in acc_conflicts if x in target_ids]
    other_filtered = [x for x in other if x["ac"] in target_ids]

    with open(output_file, "w", encoding="utf-8") as f:
        f.write("Конфликтные:\n")
        for item in conflict_filtered:
            f.write(f"{item}\n")

        f.write("\nУспешные:\n")
        for item in acc_successful:
            f.write(f"{item}\n")

        f.write("\nНе найденные:\n")
        for item in not_found:
            f.write(f"{item}\n")

        f.write("\nДругие:\n")
        for item in other_filtered:
            f.write(f"{item}\n")

    return successful_df


In [135]:
csv_file='interpro_выдача/actual_for_191125/predicted_orfs_after_interpro_new/second_try/not_annot_targetCDS_new_orf_results_coords.csv'
output_file= 'interpro_выдача/actual_for_191125/predicted_orfs_after_interpro_new/second_try/findings_notannot_target_CDS.txt'
df_notannot_target_CDS = analyze_findings(csv_file,diff,output_file)

In [137]:
csv_file='interpro_выдача/actual_for_191125/predicted_orfs_after_interpro_new/second_try/not_annot_CDS_nucl_ORFs_merged_output_orf_result_coords.csv'
output_file= 'interpro_выдача/actual_for_191125/predicted_orfs_after_interpro_new/second_try/findings_notannot_CDS.txt'
df_notannot_CDS = analyze_findings(csv_file, notannotCDS_ids, output_file)

In [138]:
def write_coords(csv_file, output_file, df):
    df_table = pd.read_csv(csv_file, sep=',', dtype=str)
    first_col = df_table.columns[0]
    df_table[first_col] = df_table[first_col].astype(str).str.strip()
    df['ac'] = df['ac'].astype(str).str.strip()
    for _, row in df.iterrows():
        ac = row['ac'].strip()          
        orf_col = row['orf'].strip()    
        coord = row['Coord'].strip()    
        strand = row.get('Strand', '+').strip()  
        mask = df_table[first_col] == ac
        if mask.any() and orf_col in df_table.columns:
            df_table.loc[mask, orf_col] = coord
            df_table.loc[mask, f'{orf_col}-strand'] = 1 if strand == '+' else -1
    df_table.to_csv(output_file, index=False, sep=',', encoding='utf-8')
    

In [142]:
csv_file = 'Astroviridae_15102025_orf-coords.csv'
output_file_1 = 'Astroviridae_26112025_1_orf-coords.csv'
write_coords (csv_file, output_file_1, df_notannot_target_CDS)

In [143]:
output_file_2 = 'Astroviridae_26112025_orf-coords.csv'
write_coords (output_file_1, output_file_2, df_notannot_CDS)